# B Cell Mapping Visualization - v1.2 COMPLETED

**Purpose**: Debug and improve visualizations from scArches L2 mapping  
**Version**: v1.2 (missing validation/report modules completed)  
**Date**: 2026-03-30  
**Author**: r2end + Copilot

---

## Fixes Applied

**P0 Critical Fixes:**
1. ✅ Global categories + fixed color palette (consistent colors across ref/query)
2. ✅ Use `astype('string')` instead of `astype(str)` (preserve NA properly)
3. ✅ Explicit marker gene source (use_raw vs layer)

**P1 Strong Recommendations:**
4. ✅ Use pandas Series masks (not numpy arrays)
5. ✅ PDF rasterization for large scatter plots
6. ✅ Improved legend strategies
7. ✅ Updated categorical dtype handling with `CategoricalDtype`

**New in v1.2:**
8. ✅ Input validation + automatic UMAP basis fallback
9. ✅ UMAP space validation figure for reference/query
10. ✅ Persisted text report + JSON manifest export

---

## Section 1: Configuration and Setup

In [ ]:
# ===== Imports =====
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
import scanpy as sc
import json

print(f"scanpy: {sc.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

In [ ]:
# ===== P1 Fix: PDF rasterization settings =====
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.settings.set_figure_params(vector_friendly=False)

print("PDF rasterization enabled for large scatter plots")

In [ ]:
# ===== Configuration =====

NOTEBOOK_VERSION = "v1.3-generic"
RUN_TIMESTAMP = datetime.now().isoformat(timespec="seconds")
PIPELINE_NAME = "generic_mapping_visualization"
RUN_MODE = "predict"  # predict | update
LABEL_LEVEL = "L2"
SPLIT_COLUMN_PREFIX = LABEL_LEVEL.lower()

# Input files
QUERY_H5AD = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad"
MERGED_H5AD = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/reference_plus_query_merged_L2.h5ad"

# Output directory
OUTPUT_DIR = Path("/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/figures_v1_3_generic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Generic key registry (treeArches-ready)
KEYS = {
    "ref_label": "Cell_Type_L2",
    "pred_label": "Cell_Type_L2_pred",
    "final_label": "Cell_Type_L2_final",
    "confidence": "mapping_confidence",
    "datasource": "data_source",
    "batch": "sample",
    "tissue": "tissue",
    "hier_pred": "tree_pred",
    "hier_updated": "tree_updated_label",
    "rejected": "tree_rejected",
    "rejection_reason": "tree_rejection_reason",
    "novelty_score": "tree_novelty_score",
    "novelty_flag": "tree_novelty_flag",
}

# Backward-compatible aliases for existing cells
L2_KEY = KEYS["ref_label"]
L2_PRED_KEY = KEYS["pred_label"]
L2_FINAL_KEY = KEYS["final_label"]
CONFIDENCE_KEY = KEYS["confidence"]
DATASOURCE_KEY = KEYS["datasource"]
BATCH_KEY = KEYS["batch"]
TISSUE_KEY = KEYS["tissue"]

# Visualization / marker presets
MARKER_SETS = {
    "bcell_core": ["CD19", "MS4A1", "CD27", "IGHD", "IGHM", "MZB1", "SDC1", "JCHAIN"],
}

# Plotting params
DPI = 300
FIGURE_FORMAT = "pdf"
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIGURE_FORMAT)

# Color palettes (will be updated after building global categories)
PALETTE_SOURCE = {"reference": "#1f77b4", "query": "#ff7f0e"}
CMAP_CONFIDENCE = "viridis"
CMAP_EXPRESSION = "Reds"
CMAP_NOVELTY = "magma"

# Runtime artifacts (keep non-Anndata outputs out of obs/uns when possible)
ARTIFACTS = {
    "adata_query": None,
    "adata_ref": None,
    "adata_merged": None,
    "latent_key": None,
    "tree": None,
    "classifier": None,
    "prediction_table": None,
    "novelty_table": None,
    "metadata": {},
}

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Run mode: {RUN_MODE}")
print(f"Label level: {LABEL_LEVEL}")
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Query H5AD: {QUERY_H5AD}")
print(f"Merged H5AD: {MERGED_H5AD}")
print(f"Registered keys: {sorted(KEYS)}")

## Section 2: Load and Inspect Data

In [ ]:
# ===== Load Query Data =====

print("Loading query data...")
adata_query = sc.read_h5ad(QUERY_H5AD)

print(f"\nQuery shape: {adata_query.shape}")
print(f"\nQuery .obs columns:")
print(adata_query.obs.columns.tolist())
print(f"\nQuery .obsm keys:")
print(list(adata_query.obsm.keys()))
print(f"\nQuery .layers keys:")
print(list(adata_query.layers.keys()) if adata_query.layers else "None")
print(f"\nQuery .raw:")
print(f"  Exists: {adata_query.raw is not None}")
if adata_query.raw is not None:
    print(f"  Shape: {adata_query.raw.shape}")

In [ ]:
# ===== Load Merged Data =====

print("Loading merged data...")
adata_merged = sc.read_h5ad(MERGED_H5AD)

print(f"\nMerged shape: {adata_merged.shape}")
print(f"\nMerged .obs columns:")
print(adata_merged.obs.columns.tolist())
print(f"\nMerged .obsm keys:")
print(list(adata_merged.obsm.keys()))
print(f"\nMerged .layers keys:")
print(list(adata_merged.layers.keys()) if adata_merged.layers else "None")
print(f"\nMerged .raw:")
print(f"  Exists: {adata_merged.raw is not None}")
if adata_merged.raw is not None:
    print(f"  Shape: {adata_merged.raw.shape}")

In [ ]:
# ===== Inspect Key Columns =====

print("=" * 60)
print("KEY COLUMNS INSPECTION")
print("=" * 60)

inspect_cols = [
    KEYS["datasource"],
    KEYS["ref_label"],
    KEYS["final_label"],
    KEYS["confidence"],
    KEYS["hier_pred"],
    KEYS["hier_updated"],
    KEYS["rejected"],
    KEYS["novelty_score"],
]
available_inspect_cols = [c for c in inspect_cols if c in adata_merged.obs.columns]

if available_inspect_cols:
    print("\nMerged .obs sample (first 10 rows):")
    print(adata_merged.obs[available_inspect_cols].head(10))

if KEYS["datasource"] in adata_merged.obs.columns:
    print("\nData source distribution:")
    print(adata_merged.obs[KEYS["datasource"]].value_counts())

tree_obs_cols = [
    KEYS["hier_pred"],
    KEYS["hier_updated"],
    KEYS["rejected"],
    KEYS["rejection_reason"],
    KEYS["novelty_score"],
    KEYS["novelty_flag"],
]
available_tree_obs_cols = [c for c in tree_obs_cols if c in adata_query.obs.columns or c in adata_merged.obs.columns]
print(f"\nTreeArches-ready fields detected: {available_tree_obs_cols if available_tree_obs_cols else 'None'}")

## Section 2.5: Validation and UMAP Setup (NEW)

## Section 3: P0 Fix #1 - Build Global Categories + Fixed Palette

In [ ]:
# ===== NEW: Validate inputs and ensure UMAP plotting basis =====

print("=" * 70)
print("VALIDATION + UMAP SETUP")
print("=" * 70)

def ensure_umap_basis(adata, adata_name, preferred_keys=("X_umap", "X_umap_scanvi", "X_umap_scvi")):
    """Ensure plotting can use adata.obsm['X_umap']; reuse a fallback if needed."""
    for key in preferred_keys:
        if key in adata.obsm:
            if key != "X_umap":
                adata.obsm["X_umap"] = np.asarray(adata.obsm[key]).copy()
                print(f"  [INFO] {adata_name}: using obsm['{key}'] as plotting basis -> obsm['X_umap']")
            else:
                print(f"  [OK] {adata_name}: found obsm['X_umap']")
            return key
    raise KeyError(
        f"{adata_name} is missing a usable UMAP basis. Checked: {preferred_keys}"
    )

def validate_obs_keys(adata, adata_name, required_keys):
    missing = [k for k in required_keys if k not in adata.obs.columns]
    if missing:
        raise KeyError(f"{adata_name} missing required obs columns: {missing}")
    print(f"  [OK] {adata_name}: required obs columns present -> {required_keys}")

def find_first_available_key(adata, candidate_keys, adata_name, required=True):
    for key in candidate_keys:
        if key and key in adata.obs.columns:
            return key
    if required:
        raise KeyError(f"{adata_name} missing all candidate keys: {candidate_keys}")
    return None

QUERY_UMAP_SOURCE = ensure_umap_basis(
    adata_query,
    "adata_query",
    preferred_keys=("X_umap", "X_umap_scanvi", "X_umap_scvi")
)
MERGED_UMAP_SOURCE = ensure_umap_basis(
    adata_merged,
    "adata_merged",
    preferred_keys=("X_umap", "X_umap_scanvi", "X_umap_scvi")
)

ACTIVE_QUERY_LABEL_KEY = find_first_available_key(
    adata_query,
    [KEYS["hier_updated"], KEYS["final_label"], KEYS["pred_label"]],
    "adata_query",
)
ACTIVE_MERGED_QUERY_LABEL_KEY = find_first_available_key(
    adata_merged,
    [KEYS["hier_updated"], KEYS["final_label"], KEYS["pred_label"]],
    "adata_merged",
)
ACTIVE_REFERENCE_LABEL_KEY = find_first_available_key(
    adata_merged,
    [KEYS["ref_label"]],
    "adata_merged",
)
ACTIVE_CONFIDENCE_KEY = find_first_available_key(
    adata_query,
    [KEYS["novelty_score"], KEYS["confidence"]],
    "adata_query",
    required=False,
) or KEYS["confidence"]

validate_obs_keys(
    adata_query,
    "adata_query",
    [ACTIVE_QUERY_LABEL_KEY],
)
validate_obs_keys(
    adata_merged,
    "adata_merged",
    [KEYS["datasource"], ACTIVE_REFERENCE_LABEL_KEY, ACTIVE_MERGED_QUERY_LABEL_KEY],
)

# Standardize source categories early (v1.3-generic: explicit categorical validation)
adata_merged.obs[KEYS["datasource"]] = adata_merged.obs[KEYS["datasource"]].astype("string")
source_dtype = CategoricalDtype(categories=["reference", "query"], ordered=False)
unexpected_sources = sorted(
    set(adata_merged.obs[KEYS["datasource"]].dropna().unique()) - set(source_dtype.categories)
)
if unexpected_sources:
    print(f"  [WARN] Unexpected data_source labels found: {unexpected_sources}")
adata_merged.obs[KEYS["datasource"]] = adata_merged.obs[KEYS["datasource"]].astype("category")

ref_mask_check = adata_merged.obs[KEYS["datasource"]].eq("reference")
qry_mask_check = adata_merged.obs[KEYS["datasource"]].eq("query")
print(f"  Reference cells in merged: {ref_mask_check.sum():,}")
print(f"  Query cells in merged: {qry_mask_check.sum():,}")

if ref_mask_check.sum() == 0 or qry_mask_check.sum() == 0:
    raise ValueError("Merged object must contain both reference and query cells.")

TREE_STATUS_KEYS = {
    role: key for role, key in {
        "hier_pred": KEYS["hier_pred"],
        "hier_updated": KEYS["hier_updated"],
        "rejected": KEYS["rejected"],
        "rejection_reason": KEYS["rejection_reason"],
        "novelty_score": KEYS["novelty_score"],
        "novelty_flag": KEYS["novelty_flag"],
    }.items()
    if key in adata_query.obs.columns or key in adata_merged.obs.columns
}
print(f"  TreeArches-ready status keys: {TREE_STATUS_KEYS if TREE_STATUS_KEYS else 'None detected'}")

ARTIFACTS["adata_query"] = adata_query
ARTIFACTS["adata_merged"] = adata_merged
ARTIFACTS["metadata"].update({
    "query_umap_source": QUERY_UMAP_SOURCE,
    "merged_umap_source": MERGED_UMAP_SOURCE,
    "active_query_label_key": ACTIVE_QUERY_LABEL_KEY,
    "active_merged_query_label_key": ACTIVE_MERGED_QUERY_LABEL_KEY,
    "active_reference_label_key": ACTIVE_REFERENCE_LABEL_KEY,
    "tree_status_keys": TREE_STATUS_KEYS,
})

print("[OK] Validation complete")

In [ ]:
# ===== P0 Fix #1: Global categories for consistent colors =====

def build_global_categories(adata, ref_key, qry_key, datasource_key):
    """Build unified category list from both reference and query labels."""
    ref_mask = adata.obs[datasource_key].eq("reference")
    qry_mask = adata.obs[datasource_key].eq("query")

    ref_vals = adata.obs.loc[ref_mask, ref_key].astype("string")
    qry_vals = adata.obs.loc[qry_mask, qry_key].astype("string")

    ref_cats = pd.Index(ref_vals.dropna().unique())
    qry_cats = pd.Index(qry_vals.dropna().unique())
    all_cats = ref_cats.union(qry_cats)

    return sorted(all_cats.tolist())

def make_palette(categories):
    """Create fixed color palette for cell types."""
    n_cats = len(categories)
    if n_cats <= 20:
        colors = list(plt.get_cmap("tab20").colors)[:n_cats]
    else:
        colors = sc.pl.palettes.default_102[:n_cats]
    return dict(zip(categories, colors))

def build_artifacts_registry(**kwargs):
    registry = {
        "adata_query": kwargs.get("adata_query"),
        "adata_ref": kwargs.get("adata_ref"),
        "adata_merged": kwargs.get("adata_merged"),
        "latent_key": kwargs.get("latent_key"),
        "tree": kwargs.get("tree"),
        "classifier": kwargs.get("classifier"),
        "prediction_table": kwargs.get("prediction_table"),
        "novelty_table": kwargs.get("novelty_table"),
        "metadata": kwargs.get("metadata", {}),
    }
    return registry

print("Building global cell type categories...")
GLOBAL_CATS = build_global_categories(
    adata_merged,
    ref_key=ACTIVE_REFERENCE_LABEL_KEY,
    qry_key=ACTIVE_MERGED_QUERY_LABEL_KEY,
    datasource_key=KEYS["datasource"],
)
GLOBAL_CAT_DTYPE = CategoricalDtype(categories=GLOBAL_CATS, ordered=False)
print(f"\nTotal unique labels: {len(GLOBAL_CATS)}")
print(f"Categories: {GLOBAL_CATS}")

CT_PALETTE = make_palette(GLOBAL_CATS)
print(f"\nPalette created with {len(CT_PALETTE)} colors")

ARTIFACTS = build_artifacts_registry(
    adata_query=adata_query,
    adata_merged=adata_merged,
    metadata={
        **ARTIFACTS.get("metadata", {}),
        "label_level": LABEL_LEVEL,
        "run_mode": RUN_MODE,
        "global_categories": GLOBAL_CATS,
        "keys": KEYS,
    },
)

viz_config = {
    "pipeline_name": PIPELINE_NAME,
    "global_categories": GLOBAL_CATS,
    "n_categories": len(GLOBAL_CATS),
    "palette_type": "tab20" if len(GLOBAL_CATS) <= 20 else "default_102",
    "version": NOTEBOOK_VERSION,
    "run_timestamp": RUN_TIMESTAMP,
    "run_mode": RUN_MODE,
    "label_level": LABEL_LEVEL,
    "query_umap_source": QUERY_UMAP_SOURCE,
    "merged_umap_source": MERGED_UMAP_SOURCE,
    "keys": KEYS,
    "tree_status_keys": TREE_STATUS_KEYS,
}

config_path = OUTPUT_DIR / "visualization_config.json"
with open(config_path, 'w') as f:
    json.dump(viz_config, f, indent=2)
print(f"\nConfig saved: {config_path}")

## Section 4: Data Cleaning with Fixed dtypes

In [ ]:
# ===== Create Cleaned Visualization Columns =====

print("Creating cleaned visualization columns...")

REF_ONLY_COL = f"{SPLIT_COLUMN_PREFIX}_reference_only"
QRY_ONLY_COL = f"{SPLIT_COLUMN_PREFIX}_query_only"
QRY_CONF_ONLY_COL = f"{SPLIT_COLUMN_PREFIX}_confidence_query_only"

def prepare_split_obs_columns(
    adata,
    ref_key,
    qry_key,
    datasource_key,
    confidence_key=None,
    categories=None,
    prefix="label",
):
    """Create reference-only/query-only/confidence split columns for visualization."""
    ref_mask = adata.obs[datasource_key].eq("reference")
    qry_mask = adata.obs[datasource_key].eq("query")

    ref_only_col = f"{prefix}_reference_only"
    qry_only_col = f"{prefix}_query_only"
    qry_conf_col = f"{prefix}_confidence_query_only"

    adata.obs[qry_only_col] = pd.Series(pd.NA, index=adata.obs_names, dtype="string")
    adata.obs[ref_only_col] = pd.Series(pd.NA, index=adata.obs_names, dtype="string")

    if qry_key in adata.obs.columns:
        qry_ser = adata.obs[qry_key].astype("string")
        adata.obs.loc[qry_mask, qry_only_col] = qry_ser[qry_mask]

    if ref_key in adata.obs.columns:
        ref_ser = adata.obs[ref_key].astype("string")
        adata.obs.loc[ref_mask, ref_only_col] = ref_ser[ref_mask]

    if categories is not None:
        dtype = CategoricalDtype(categories=categories, ordered=False)
        adata.obs[qry_only_col] = adata.obs[qry_only_col].astype("string").astype(dtype)
        adata.obs[ref_only_col] = adata.obs[ref_only_col].astype("string").astype(dtype)

    conf_vals = np.full(adata.n_obs, np.nan, dtype=float)
    if confidence_key and confidence_key in adata.obs.columns:
        conf_qry = pd.to_numeric(adata.obs.loc[qry_mask, confidence_key], errors='coerce').to_numpy()
        conf_vals[qry_mask] = conf_qry
    adata.obs[qry_conf_col] = conf_vals

    return {
        "ref_mask": ref_mask,
        "qry_mask": qry_mask,
        "ref_only_col": ref_only_col,
        "qry_only_col": qry_only_col,
        "qry_conf_col": qry_conf_col,
    }

split_info = prepare_split_obs_columns(
    adata_merged,
    ref_key=ACTIVE_REFERENCE_LABEL_KEY,
    qry_key=ACTIVE_MERGED_QUERY_LABEL_KEY,
    datasource_key=KEYS["datasource"],
    confidence_key=KEYS["confidence"],
    categories=GLOBAL_CATS,
    prefix=SPLIT_COLUMN_PREFIX,
)

ref_mask = split_info["ref_mask"]
qry_mask = split_info["qry_mask"]

print(f"Reference cells: {ref_mask.sum():,}")
print(f"Query cells: {qry_mask.sum():,}")
print(f"  {split_info['qry_only_col']} created -> non-NA: {adata_merged.obs[QRY_ONLY_COL].notna().sum():,}")
print(f"  {split_info['ref_only_col']} created -> non-NA: {adata_merged.obs[REF_ONLY_COL].notna().sum():,}")
print(f"  {split_info['qry_conf_col']} created -> finite: {np.isfinite(adata_merged.obs[QRY_CONF_ONLY_COL]).sum():,}")

for bool_role in ["rejected", "novelty_flag"]:
    key = KEYS[bool_role]
    if key in adata_query.obs.columns:
        adata_query.obs[key] = adata_query.obs[key].astype("boolean")
    if key in adata_merged.obs.columns:
        adata_merged.obs[key] = adata_merged.obs[key].astype("boolean")

print("\nColumn creation complete!")

In [ ]:
# ===== Similarly prepare query-only object =====

print("Preparing query object with global categories...")

for key_name in ["final_label", "pred_label", "hier_pred", "hier_updated"]:
    obs_key = KEYS[key_name]
    if obs_key in adata_query.obs.columns:
        adata_query.obs[obs_key] = (
            adata_query.obs[obs_key]
            .astype("string")
            .astype(GLOBAL_CAT_DTYPE)
        )
        print(f"  {obs_key} updated with global categories")

for bool_role in ["rejected", "novelty_flag"]:
    obs_key = KEYS[bool_role]
    if obs_key in adata_query.obs.columns:
        adata_query.obs[obs_key] = adata_query.obs[obs_key].astype("boolean")
        print(f"  {obs_key} normalized to pandas BooleanDtype")

print("Done!")

## Section 5: P0 Fix #3 - Determine Marker Gene Source

In [ ]:
# ===== P0 Fix #3: Determine how to access marker genes =====

def determine_marker_source(adata):
    """
    Determine best source for marker gene expression.
    
    Returns:
        tuple: (use_raw, layer)
    """
    # Priority 1: .raw with full genes
    if adata.raw is not None and adata.raw.n_vars > adata.n_vars:
        return True, None
    
    # Priority 2: log1p layer
    if adata.layers and "log1p" in adata.layers:
        return False, "log1p"
    
    # Priority 3: counts layer (will need normalization)
    if adata.layers and "counts" in adata.layers:
        return False, "counts"
    
    # Fallback: use .X
    return False, None


# Check query
print("Query dataset:")
use_raw_query, layer_query = determine_marker_source(adata_query)
print(f"  use_raw: {use_raw_query}")
print(f"  layer: {layer_query}")

# Check merged
print("\nMerged dataset:")
use_raw_merged, layer_merged = determine_marker_source(adata_merged)
print(f"  use_raw: {use_raw_merged}")
print(f"  layer: {layer_merged}")

# Store for later use
MARKER_CONFIG = {
    "query": {"use_raw": use_raw_query, "layer": layer_query},
    "merged": {"use_raw": use_raw_merged, "layer": layer_merged}
}

In [ ]:
# ===== Helper function for marker availability =====

def get_available_markers(adata, marker_list, use_raw=False):
    """
    Check which markers are available in the dataset.
    
    Args:
        adata: AnnData object
        marker_list: List of gene names
        use_raw: Whether to check .raw.var_names
    
    Returns:
        list: Available marker genes
    """
    if use_raw and adata.raw is not None:
        var_names = adata.raw.var_names
    else:
        var_names = adata.var_names
    
    return [g for g in marker_list if g in var_names]


# Test with B cell markers
test_markers = ["CD19", "MS4A1", "CD27", "IGHD", "MZB1", "SDC1", "JCHAIN"]

print("Available markers in query:")
avail_query = get_available_markers(
    adata_query, 
    test_markers, 
    use_raw=use_raw_query
)
print(f"  {avail_query}")

print("\nAvailable markers in merged:")
avail_merged = get_available_markers(
    adata_merged, 
    test_markers, 
    use_raw=use_raw_merged
)
print(f"  {avail_merged}")

## Section 6: Helper Functions for Plotting

In [ ]:
# ===== P1 Fix #5: Rasterization helper =====

def save_rasterized_figure(fig, path, dpi=300, rasterize_scatter=True):
    """
    Save figure with scatter plots rasterized to reduce file size.
    
    Args:
        fig: matplotlib figure
        path: output path
        dpi: resolution
        rasterize_scatter: whether to rasterize scatter collections
    """
    if rasterize_scatter:
        for ax in fig.axes:
            for coll in ax.collections:
                coll.set_rasterized(True)
    
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


print("Helper functions defined")

## Section 7: Query-Only Visualizations (FIXED)

In [ ]:
# ===== Query Overview: 4-Panel Figure (FIXED) =====

fig, axes = plt.subplots(2, 2, figsize=(18, 18))

# Panel 1: Raw predictions (all cells) - FIXED palette
if L2_PRED_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query, 
        color=L2_PRED_KEY,
        ax=axes[0, 0],
        show=False,
        title="L2 Predictions (All Cells)",
        legend_loc="right margin",
        palette=CT_PALETTE,  # P0 Fix #1
        frameon=False,
        s=30
    )

# Panel 2: Filtered predictions - FIXED palette
if L2_FINAL_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query,
        color=L2_FINAL_KEY,
        ax=axes[0, 1],
        show=False,
        title="L2 Final (Confidence >= 0.5)",
        legend_loc="right margin",
        palette=CT_PALETTE,  # P0 Fix #1
        frameon=False,
        s=30
    )

# Panel 3: Confidence score heatmap
if CONFIDENCE_KEY in adata_query.obs.columns:
    sc.pl.umap(
        adata_query,
        color=CONFIDENCE_KEY,
        ax=axes[1, 0],
        show=False,
        title="Mapping Confidence Score",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        frameon=False,
        s=30
    )

# Panel 4: Confidence histogram
if CONFIDENCE_KEY in adata_query.obs.columns:
    conf_vals = pd.to_numeric(adata_query.obs[CONFIDENCE_KEY], errors='coerce').to_numpy()
    conf_vals = conf_vals[np.isfinite(conf_vals)]
    
    axes[1, 1].hist(
        conf_vals, 
        bins=50, 
        edgecolor='black', 
        alpha=0.7,
        color='steelblue'
    )
    axes[1, 1].axvline(
        0.5, 
        color='red', 
        linestyle='--', 
        linewidth=2, 
        label='Threshold = 0.5'
    )
    axes[1, 1].set_xlabel('Mapping Confidence', fontsize=12)
    axes[1, 1].set_ylabel('Number of Cells', fontsize=12)
    axes[1, 1].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=11)
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].spines['top'].set_visible(False)
    axes[1, 1].spines['right'].set_visible(False)

plt.tight_layout()
output_path = OUTPUT_DIR / f"query_overview_fixed.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"Saved: {output_path}")

In [ ]:
# ===== Query Cell Type Proportions =====

if L2_FINAL_KEY in adata_query.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # Bar plot
    ct_counts = adata_query.obs[L2_FINAL_KEY].value_counts()
    ct_counts.plot(
        kind='bar',
        ax=axes[0],
        color='steelblue',
        edgecolor='black'
    )
    axes[0].set_title('Cell Type Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cell Type', fontsize=12)
    axes[0].set_ylabel('Number of Cells', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(ct_counts.values):
        axes[0].text(i, v + max(ct_counts.values)*0.01, f'{v:,}', 
                     ha='center', va='bottom', fontsize=10)
    
    # Pie chart
    ct_pct = adata_query.obs[L2_FINAL_KEY].value_counts(normalize=True) * 100
    
    # Use colors from global palette
    pie_colors = [CT_PALETTE.get(ct, 'gray') for ct in ct_pct.index]
    
    axes[1].pie(
        ct_pct.values,
        labels=ct_pct.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=pie_colors,  # P0 Fix #1
        textprops={'fontsize': 10}
    )
    axes[1].set_title('Cell Type Distribution (Percentage)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"query_celltype_proportions.{FIGURE_FORMAT}"
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {output_path}")

## Section 8: Merged Visualizations (FIXED)

In [ ]:
# ===== Merged Overview: generic mapping figure =====

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

# Panel 1: Data source
sc.pl.umap(
    adata_merged,
    color=KEYS["datasource"],
    ax=axes[0, 0],
    show=False,
    title="Data Source",
    palette=PALETTE_SOURCE,
    frameon=False,
    s=20
)

# Panel 2: Reference labels
sc.pl.umap(
    adata_merged,
    color=REF_ONLY_COL,
    ax=axes[0, 1],
    show=False,
    title=f"Reference {LABEL_LEVEL} Labels (Reference Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 3: Query labels / predictions
sc.pl.umap(
    adata_merged,
    color=QRY_ONLY_COL,
    ax=axes[0, 2],
    show=False,
    title=f"Query {LABEL_LEVEL} Labels (Query Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 4: Batch
if KEYS["batch"] in adata_merged.obs.columns:
    sc.pl.umap(
        adata_merged,
        color=KEYS["batch"],
        ax=axes[1, 0],
        show=False,
        title="Batch",
        frameon=False,
        s=20,
        legend_loc=None
    )
else:
    axes[1, 0].axis('off')

# Panel 5: Confidence (query only)
sc.pl.umap(
    adata_merged,
    color=QRY_CONF_ONLY_COL,
    ax=axes[1, 1],
    show=False,
    title="Mapping Confidence (Query Only)",
    cmap=CMAP_CONFIDENCE,
    vmin=0,
    vmax=1,
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 6: Marker gene
marker_gene = MARKER_SETS["bcell_core"][0]
available = get_available_markers(
    adata_merged,
    [marker_gene],
    use_raw=use_raw_merged
)

if marker_gene in available:
    plot_kwargs = {
        'color': marker_gene,
        'ax': axes[1, 2],
        'show': False,
        'title': f"{marker_gene} Expression",
        'cmap': CMAP_EXPRESSION,
        'frameon': False,
        's': 20,
    }
    if use_raw_merged:
        plot_kwargs['use_raw'] = True
    elif layer_merged is not None:
        plot_kwargs['layer'] = layer_merged
    else:
        plot_kwargs['use_raw'] = False
    sc.pl.umap(adata_merged, **plot_kwargs)
else:
    axes[1, 2].text(
        0.5, 0.5,
        f"{marker_gene}\nNot Available",
        ha='center',
        va='center',
        transform=axes[1, 2].transAxes,
        fontsize=14
    )
    axes[1, 2].axis('off')

plt.tight_layout()
output_path = OUTPUT_DIR / f"merged_overview_generic.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"Saved: {output_path}")

In [ ]:
# ===== Side-by-Side Comparison: Reference vs Query / Updated Labels =====

comparison_right_key = KEYS["hier_updated"] if KEYS["hier_updated"] in adata_merged.obs.columns else QRY_ONLY_COL
comparison_right_title = (
    "Query: Hierarchy-updated Labels"
    if comparison_right_key == KEYS["hier_updated"]
    else f"Query: Predicted {LABEL_LEVEL} Labels"
)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sc.pl.umap(
    adata_merged,
    color=REF_ONLY_COL,
    ax=axes[0],
    show=False,
    title=f"Reference: Original {LABEL_LEVEL} Labels",
    legend_loc="right margin",
    palette=CT_PALETTE,
    frameon=False,
    s=25,
    na_color='lightgray'
)

sc.pl.umap(
    adata_merged,
    color=comparison_right_key,
    ax=axes[1],
    show=False,
    title=comparison_right_title,
    legend_loc="right margin",
    palette=CT_PALETTE,
    frameon=False,
    s=25,
    na_color='lightgray'
)

plt.tight_layout()
output_path = OUTPUT_DIR / f"reference_vs_query_or_updated_comparison.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"Saved: {output_path}")

## Section 8.5: UMAP Space Validation (NEW)

## Section 9: treeArches-Ready Diagnostics (NEW)

## Section 10: Marker Gene Panel (FIXED)

In [ ]:
# ===== NEW: UMAP space validation for merged reference/query =====

print("Running UMAP space validation...")

ref_mask_umap = adata_merged.obs[KEYS["datasource"]].eq("reference")
qry_mask_umap = adata_merged.obs[KEYS["datasource"]].eq("query")

umap_ref = np.asarray(adata_merged.obsm["X_umap"])[ref_mask_umap.to_numpy()]
umap_qry = np.asarray(adata_merged.obsm["X_umap"])[qry_mask_umap.to_numpy()]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].hist(umap_ref[:, 0], bins=50, alpha=0.5, density=True,
             label="Reference", color=PALETTE_SOURCE["reference"])
axes[0].hist(umap_qry[:, 0], bins=50, alpha=0.5, density=True,
             label="Query", color=PALETTE_SOURCE["query"])
axes[0].set_xlabel("UMAP X", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("UMAP X Distribution", fontsize=14, fontweight='bold')
axes[0].legend(frameon=True)
axes[0].grid(alpha=0.3)

axes[1].hist(umap_ref[:, 1], bins=50, alpha=0.5, density=True,
             label="Reference", color=PALETTE_SOURCE["reference"])
axes[1].hist(umap_qry[:, 1], bins=50, alpha=0.5, density=True,
             label="Query", color=PALETTE_SOURCE["query"])
axes[1].set_xlabel("UMAP Y", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("UMAP Y Distribution", fontsize=14, fontweight='bold')
axes[1].legend(frameon=True)
axes[1].grid(alpha=0.3)

axes[2].scatter(umap_ref[:, 0], umap_ref[:, 1], s=3, alpha=0.25,
                c=PALETTE_SOURCE["reference"], label="Reference")
axes[2].scatter(umap_qry[:, 0], umap_qry[:, 1], s=3, alpha=0.25,
                c=PALETTE_SOURCE["query"], label="Query")
axes[2].set_xlabel("UMAP X", fontsize=12)
axes[2].set_ylabel("UMAP Y", fontsize=12)
axes[2].set_title("Reference vs Query Overlay", fontsize=14, fontweight='bold')
axes[2].legend(frameon=True)
axes[2].grid(alpha=0.3)

ref_x_range = np.ptp(umap_ref[:, 0])
ref_y_range = np.ptp(umap_ref[:, 1])
qry_x_range = np.ptp(umap_qry[:, 0])
qry_y_range = np.ptp(umap_qry[:, 1])
stats_text = (
    f"Reference range: X={ref_x_range:.2f}, Y={ref_y_range:.2f}\n"
    f"Query range:     X={qry_x_range:.2f}, Y={qry_y_range:.2f}\n"
    f"Range ratio:     X={qry_x_range / ref_x_range:.2f}, Y={qry_y_range / ref_y_range:.2f}"
    if ref_x_range > 0 and ref_y_range > 0
    else "Range ratio: N/A"
)
axes[2].text(
    0.03, 0.97, stats_text,
    transform=axes[2].transAxes,
    va='top',
    fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
)

plt.tight_layout()
output_path = OUTPUT_DIR / f"umap_space_validation.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"Saved: {output_path}")

In [ ]:
# ===== NEW: treeArches-ready diagnostic plots =====

print("Running treeArches-ready diagnostics...")

tree_diag_specs = [
    (KEYS["rejected"], "Rejected vs Accepted", None, None),
    (KEYS["novelty_score"], "Novelty Score", CMAP_NOVELTY, (0, 1)),
    (KEYS["hier_updated"], "Hierarchy-updated Label", CT_PALETTE, None),
]
available_tree_diag = [spec for spec in tree_diag_specs if spec[0] in adata_query.obs.columns or spec[0] in adata_merged.obs.columns]

if available_tree_diag:
    fig, axes = plt.subplots(1, len(available_tree_diag), figsize=(7 * len(available_tree_diag), 6))
    axes = np.atleast_1d(axes)

    for ax, (obs_key, title, palette_or_cmap, limits) in zip(axes, available_tree_diag):
        plot_kwargs = {
            "adata": adata_query if obs_key in adata_query.obs.columns else adata_merged,
            "color": obs_key,
            "ax": ax,
            "show": False,
            "title": title,
            "frameon": False,
            "s": 25,
        }
        if obs_key == KEYS["hier_updated"]:
            plot_kwargs["palette"] = CT_PALETTE
            plot_kwargs["na_color"] = "lightgray"
        elif obs_key == KEYS["rejected"]:
            plot_kwargs["palette"] = {False: "#4daf4a", True: "#e41a1c"}
            plot_kwargs["na_color"] = "lightgray"
        elif obs_key == KEYS["novelty_score"]:
            plot_kwargs["cmap"] = palette_or_cmap
            plot_kwargs["vmin"] = limits[0]
            plot_kwargs["vmax"] = limits[1]

        sc.pl.umap(**plot_kwargs)

    plt.tight_layout()
    output_path = OUTPUT_DIR / f"treearches_ready_diagnostics.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    print(f"Saved: {output_path}")
else:
    print("No treeArches-specific obs fields detected; skipping tree diagnostics.")

In [ ]:
# ===== Marker Genes Panel (generic marker set) =====

marker_genes = MARKER_SETS["bcell_core"]

available_markers = get_available_markers(
    adata_query,
    marker_genes,
    use_raw=use_raw_query
)

print(f"Available markers: {available_markers}")

if len(available_markers) > 0:
    n_markers = len(available_markers)
    ncols = 4
    nrows = int(np.ceil(n_markers / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    base_kwargs = {
        'show': False,
        'cmap': CMAP_EXPRESSION,
        'frameon': False,
        's': 30,
    }

    if use_raw_query:
        base_kwargs['use_raw'] = True
    elif layer_query is not None:
        base_kwargs['layer'] = layer_query
    else:
        base_kwargs['use_raw'] = False

    for i, gene in enumerate(available_markers):
        sc.pl.umap(
            adata_query,
            color=gene,
            ax=axes[i],
            title=f"{gene} Expression",
            **base_kwargs
        )

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    output_path = OUTPUT_DIR / f"marker_panel_{SPLIT_COLUMN_PREFIX}.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    print(f"Saved: {output_path}")
else:
    print("No marker genes found in dataset.")

## Section 10: Quality Control (P2 optimization applied)

In [ ]:
# ===== Confidence vs Cell Type (P2 optimization: top N only) =====

if CONFIDENCE_KEY in adata_query.obs.columns and L2_FINAL_KEY in adata_query.obs.columns:
    
    # Prepare data
    df_plot = adata_query.obs[[L2_FINAL_KEY, CONFIDENCE_KEY]].copy()
    df_plot[CONFIDENCE_KEY] = pd.to_numeric(df_plot[CONFIDENCE_KEY], errors='coerce')
    df_plot = df_plot.dropna()
    
    # P2 optimization: Take top 10 cell types by count
    top_cts = df_plot[L2_FINAL_KEY].value_counts().head(10).index
    df_plot_top = df_plot[df_plot[L2_FINAL_KEY].isin(top_cts)].copy()
    
    # Sort by median confidence
    ct_order = df_plot_top.groupby(L2_FINAL_KEY)[CONFIDENCE_KEY].median().sort_values(ascending=False).index
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # Violin plot
    sns.violinplot(
        data=df_plot_top,
        x=L2_FINAL_KEY,
        y=CONFIDENCE_KEY,
        order=ct_order,
        ax=axes[0],
        palette='Set2'
    )
    axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
    axes[0].set_title('Confidence by Cell Type (Top 10, Violin)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Cell Type', fontsize=12)
    axes[0].set_ylabel('Mapping Confidence', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    sns.boxplot(
        data=df_plot_top,
        x=L2_FINAL_KEY,
        y=CONFIDENCE_KEY,
        order=ct_order,
        ax=axes[1],
        palette='Set2'
    )
    axes[1].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
    axes[1].set_title('Confidence by Cell Type (Top 10, Box)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Cell Type', fontsize=12)
    axes[1].set_ylabel('Mapping Confidence', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"confidence_by_celltype_top10.{FIGURE_FORMAT}"
    plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {output_path}")

In [ ]:
# ===== Confidence Summary Table =====

if CONFIDENCE_KEY in adata_query.obs.columns and L2_FINAL_KEY in adata_query.obs.columns:
    
    df_summary = adata_query.obs.copy()
    df_summary[CONFIDENCE_KEY] = pd.to_numeric(df_summary[CONFIDENCE_KEY], errors='coerce')
    
    summary_stats = df_summary.groupby(L2_FINAL_KEY)[CONFIDENCE_KEY].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max')
    ]).round(3)
    
    # Add percentage
    summary_stats['Percentage'] = (summary_stats['Count'] / summary_stats['Count'].sum() * 100).round(2)
    
    # Sort by count
    summary_stats = summary_stats.sort_values('Count', ascending=False)
    
    print("=" * 80)
    print("CONFIDENCE SUMMARY BY CELL TYPE")
    print("=" * 80)
    print(summary_stats.to_string())
    print("=" * 80)
    
    # Save to CSV
    csv_path = OUTPUT_DIR / "confidence_summary_by_celltype.csv"
    summary_stats.to_csv(csv_path)
    print(f"\nSaved summary table: {csv_path}")

## Section 11: High-Resolution Individual Exports

In [ ]:
# ===== Export Individual UMAPs (Publication Ready) =====

print("Exporting individual high-resolution UMAPs...")

export_params = {
    'frameon': False,
    's': 50,
    'show': False
}

# 1. Query L2 Final (P1 Fix #6: on data legend)
if L2_FINAL_KEY in adata_query.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata_query,
        color=L2_FINAL_KEY,
        ax=ax,
        title="",
        legend_loc="on data",  # P1 Fix #6
        legend_fontsize=10,
        legend_fontoutline=2,
        palette=CT_PALETTE,  # P0 Fix #1
        **export_params
    )
    output_path = OUTPUT_DIR / f"umap_L2_final_highres.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
    print(f"  Saved: {output_path.name}")

# 2. Query Confidence
if CONFIDENCE_KEY in adata_query.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata_query,
        color=CONFIDENCE_KEY,
        ax=ax,
        title="",
        cmap=CMAP_CONFIDENCE,
        vmin=0,
        vmax=1,
        **export_params
    )
    output_path = OUTPUT_DIR / f"umap_confidence_highres.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
    print(f"  Saved: {output_path.name}")

# 3. Data Source (Merged)
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=DATASOURCE_KEY,
    ax=ax,
    title="",
    palette=PALETTE_SOURCE,
    **export_params
)
output_path = OUTPUT_DIR / f"umap_data_source_highres.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)  # P1 Fix #5
print(f"  Saved: {output_path.name}")

print("\nExport complete!")

## Section 12: Summary Report

In [ ]:
# ===== Generate Summary Report + Persist Outputs =====

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append(f"GENERIC MAPPING VISUALIZATION SUMMARY ({NOTEBOOK_VERSION})")
summary_lines.append("=" * 80)

summary_lines.append(f"\nPipeline: {PIPELINE_NAME}")
summary_lines.append(f"Run mode: {RUN_MODE}")
summary_lines.append(f"Label level: {LABEL_LEVEL}")

summary_lines.append("\nQuery Dataset:")
summary_lines.append(f"  Total cells: {adata_query.n_obs:,}")
summary_lines.append(f"  Total genes: {adata_query.n_vars:,}")

if ACTIVE_QUERY_LABEL_KEY in adata_query.obs.columns:
    summary_lines.append("\nActive Label Distribution:")
    for ct, count in adata_query.obs[ACTIVE_QUERY_LABEL_KEY].value_counts().head(10).items():
        pct = count / adata_query.n_obs * 100
        summary_lines.append(f"  {ct}: {count:,} ({pct:.1f}%)")

if CONFIDENCE_KEY in adata_query.obs.columns:
    conf = pd.to_numeric(adata_query.obs[CONFIDENCE_KEY], errors='coerce')
    summary_lines.append("\nMapping Confidence:")
    summary_lines.append(f"  Mean: {conf.mean():.3f}")
    summary_lines.append(f"  Median: {conf.median():.3f}")
    summary_lines.append(
        f"  High confidence (>0.9): {(conf > 0.9).sum():,} ({(conf > 0.9).mean()*100:.1f}%)"
    )
    summary_lines.append(
        f"  Low confidence (<0.5): {(conf < 0.5).sum():,} ({(conf < 0.5).mean()*100:.1f}%)"
    )

summary_lines.append("\nMerged Dataset:")
summary_lines.append(f"  Total cells: {adata_merged.n_obs:,}")
summary_lines.append(
    f"  Reference cells: {(adata_merged.obs[KEYS['datasource']] == 'reference').sum():,}"
)
summary_lines.append(
    f"  Query cells: {(adata_merged.obs[KEYS['datasource']] == 'query').sum():,}"
)

summary_lines.append("\nColor Palette:")
summary_lines.append(f"  Global categories: {len(GLOBAL_CATS)}")
summary_lines.append(f"  Palette type: {viz_config['palette_type']}")
summary_lines.append("  Consistent colors: ✅")

summary_lines.append("\nUMAP Sources:")
summary_lines.append(f"  Query: {QUERY_UMAP_SOURCE}")
summary_lines.append(f"  Merged: {MERGED_UMAP_SOURCE}")

summary_lines.append("\nMarker Gene Source:")
summary_lines.append(f"  Query: use_raw={use_raw_query}, layer={layer_query}")
summary_lines.append(f"  Merged: use_raw={use_raw_merged}, layer={layer_merged}")
summary_lines.append(f"  Available markers: {', '.join(available_markers) if len(available_markers) else 'None'}")

if TREE_STATUS_KEYS:
    summary_lines.append("\nHierarchy / treeArches Status:")
    if KEYS["rejected"] in adata_query.obs.columns:
        rejected = adata_query.obs[KEYS["rejected"]].astype("boolean")
        summary_lines.append(f"  Rejected cells: {int(rejected.fillna(False).sum()):,}")
    if KEYS["novelty_flag"] in adata_query.obs.columns:
        novelty_flag = adata_query.obs[KEYS["novelty_flag"]].astype("boolean")
        summary_lines.append(f"  Novel candidates: {int(novelty_flag.fillna(False).sum()):,}")
    if KEYS["novelty_score"] in adata_query.obs.columns:
        novelty_score = pd.to_numeric(adata_query.obs[KEYS["novelty_score"]], errors='coerce')
        summary_lines.append(f"  Novelty score mean: {novelty_score.mean():.3f}")
    if KEYS["hier_updated"] in adata_query.obs.columns:
        top_updated = adata_query.obs[KEYS["hier_updated"]].value_counts().head(5).to_dict()
        summary_lines.append(f"  Top updated labels: {top_updated}")

generated_files = sorted(p.name for p in OUTPUT_DIR.glob('*'))
summary_lines.append(f"\nOutput Directory: {OUTPUT_DIR}")
summary_lines.append(f"  Total files generated: {len(generated_files)}")
for fname in generated_files:
    summary_lines.append(f"  - {fname}")

summary_lines.append("\n" + "=" * 80)
summary_lines.append(f"IMPROVEMENTS APPLIED ({NOTEBOOK_VERSION})")
summary_lines.append("=" * 80)
summary_lines.append("  ✅ KEYS registry replaces hard-coded L2-only bindings")
summary_lines.append("  ✅ prepare_split_obs_columns() centralizes ref/query split columns")
summary_lines.append("  ✅ ARTIFACTS registry separates non-AnnData outputs from obs/uns")
summary_lines.append("  ✅ treeArches-ready fields and diagnostics are auto-detected")
summary_lines.append("  ✅ Summary/manifest now capture hierarchy-aware metadata")
summary_lines.append("\n" + "=" * 80)
summary_lines.append("GENERIC VISUALIZATION TEMPLATE READY")
summary_lines.append("=" * 80)

print("\n".join(summary_lines))

report_path = OUTPUT_DIR / "visualization_summary_report_generic.txt"
report_path.write_text("\n".join(summary_lines), encoding="utf-8")

manifest = {
    "version": NOTEBOOK_VERSION,
    "pipeline_name": PIPELINE_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "run_mode": RUN_MODE,
    "input_files": {
        "query_h5ad": QUERY_H5AD,
        "merged_h5ad": MERGED_H5AD
    },
    "shapes": {
        "query": [int(adata_query.n_obs), int(adata_query.n_vars)],
        "merged": [int(adata_merged.n_obs), int(adata_merged.n_vars)]
    },
    "umap_sources": {
        "query": QUERY_UMAP_SOURCE,
        "merged": MERGED_UMAP_SOURCE
    },
    "keys": KEYS,
    "active_keys": {
        "query_label": ACTIVE_QUERY_LABEL_KEY,
        "merged_query_label": ACTIVE_MERGED_QUERY_LABEL_KEY,
        "reference_label": ACTIVE_REFERENCE_LABEL_KEY
    },
    "global_categories": GLOBAL_CATS,
    "palette_type": viz_config["palette_type"],
    "marker_config": {
        "query": {"use_raw": use_raw_query, "layer": layer_query},
        "merged": {"use_raw": use_raw_merged, "layer": layer_merged},
        "available_markers": available_markers,
        "marker_sets": MARKER_SETS
    },
    "tree_status_keys": TREE_STATUS_KEYS,
    "artifacts_metadata": ARTIFACTS.get("metadata", {}),
    "generated_files": generated_files
}

if KEYS["rejected"] in adata_query.obs.columns:
    rejection_summary = (
        adata_query.obs.groupby(ACTIVE_QUERY_LABEL_KEY, dropna=False)[KEYS["rejected"]]
        .apply(lambda x: pd.Series(x, dtype='boolean').fillna(False).mean())
        .sort_values(ascending=False)
        .rename("rejection_rate")
    )
    rejection_summary.to_csv(OUTPUT_DIR / "tree_rejection_summary.csv")

if KEYS["novelty_score"] in adata_query.obs.columns:
    novelty_summary = (
        adata_query.obs.groupby(ACTIVE_QUERY_LABEL_KEY, dropna=False)[KEYS["novelty_score"]]
        .apply(lambda x: pd.to_numeric(x, errors='coerce').mean())
        .sort_values(ascending=False)
        .rename("mean_novelty_score")
    )
    novelty_summary.to_csv(OUTPUT_DIR / "tree_novelty_summary.csv")

manifest_path = OUTPUT_DIR / "visualization_manifest_generic.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nSaved summary report: {report_path}")
print(f"Saved manifest JSON: {manifest_path}")

---

## Summary of Fixes

### P0 Critical Fixes (Must Have)

1. **Global Categories + Fixed Palette**: Same cell type now has identical color across all plots (reference vs query comparison is no longer misleading)

2. **Proper NA Handling**: Using `astype('string')` instead of `astype(str)` prevents NA from becoming "nan" string category

3. **Explicit Marker Source**: Marker gene expression now explicitly specifies `use_raw` or `layer` to ensure correct data source

### P1 Strong Recommendations

4. **Pandas Series Masks**: Using `.eq()` instead of `.to_numpy()` for safer indexing

5. **PDF Rasterization**: Large scatter plots are now rasterized to reduce file size and prevent Illustrator crashes

6. **Legend Strategies**: "on data" for high-res exports, "right margin" or None for overviews

7. **Updated Dtype Check**: Visualization categories now use shared `CategoricalDtype` definitions

### New in v1.2

8. **Input Validation**: Added required-column checks and automatic fallback to available UMAP embeddings

9. **UMAP Space Validation**: Added reference-vs-query coordinate distribution checks and overlay figure

10. **Persisted Reporting**: Summary is now exported to both `.txt` and `.json` manifest files for reproducibility

### P2 Optimizations

11. **Top N for QC**: Confidence plots now show top 10 cell types only for better readability

12. **Config Export**: Visualization configuration saved to JSON for reproducibility

---

## Next Steps

1. **Validate Results**: Check that reference and query have consistent colors across panels

2. **Marker Validation**: Verify B cell lineage markers show expected patterns

3. **Review UMAP QC**: Confirm reference/query overlay and coordinate ranges look reasonable

4. **Archive Outputs**: Keep `visualization_summary_report_v1_2.txt` and `visualization_manifest_v1_2.json` with the figures for downstream sharing

---